In [ ]:
#!/usr/bin/env python3
"""
Frozen Full Benchmark: O(1) Analytic Dynamic-Load & Grip-Aware Swerve Control vs. SOTA ADMM-QP
Generates:
  - metrics_per_trial.csv
  - metrics_summary.json
  - benchmark_cache.npz
  - 6 high-resolution (.png) figures for publication
"""

import json
import math
import time
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

BENCHMARK_OUTPUT_DIR = Path("../benchmark_data")
FIGURE_OUTPUT_DIR = Path("../../docs/images/benchmark_suite")
BENCHMARK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ==============================================================================
# 1. PHYSICAL CONSTANTS AND FROZEN EXPERIMENTAL CONFIGURATION
# ==============================================================================
DT_CONTROL = 0.002       # 500 Hz control period
CONTROL_FREQ_HZ = 500
N_SUBSTEPS = 10          # 5 kHz physics integration substeps
G = 9.81
TRIALS_PER_CLASS = 40    # N = 120 paired trials x 4 controllers = 480 episodes
T_STEPS_EPISODE = 950    # 1.90 s per trial
SEED_MASTER = 2026


@dataclass(frozen=True)
class RobotSpec:
    name: str
    short_label: str
    m_nom: float
    Iz_nom: float
    lx: float
    ly: float
    h_cg: float
    r_w: float
    j_w: float
    f_stall: float
    v_free: float
    tau_drive: float
    tau_steer: float
    max_steer_rate: float
    drag_lin: float
    drag_yaw: float
    v_cruise_range: Tuple[float, float]
    amp_y_range: Tuple[float, float]

    @property
    def corners(self) -> Tuple[Tuple[float, float], ...]:
        return (
            (+self.lx, +self.ly),
            (+self.lx, -self.ly),
            (-self.lx, +self.ly),
            (-self.lx, -self.ly),
        )

    @property
    def r_sq(self) -> float:
        return self.lx * self.lx + self.ly * self.ly


ROBOT_CLASSES: Tuple[RobotSpec, ...] = (
    RobotSpec(
        "Class1_AgileMini_32kg", "Agile Mini\n(32 kg)",
        32.0, 1.65, 0.21, 0.21, 0.16, 0.038, 0.008,
        220.0, 5.50, 0.006, 0.009, 24.0, 2.5, 0.40,
        (2.1, 2.6), (0.40, 0.56),
    ),
    RobotSpec(
        "Class2_Standard_72kg", "Standard FRC\n(72 kg)",
        72.0, 5.20, 0.29, 0.29, 0.24, 0.0508, 0.018,
        320.0, 4.80, 0.010, 0.014, 18.0, 4.5, 0.85,
        (2.0, 2.5), (0.45, 0.65),
    ),
    RobotSpec(
        "Class3_HeavyAMR_145kg", "Heavy AMR\n(145 kg)",
        145.0, 16.5, 0.42, 0.38, 0.34, 0.076, 0.055,
        480.0, 3.80, 0.016, 0.022, 13.0, 8.0, 1.80,
        (1.6, 2.0), (0.45, 0.62),
    ),
)

CONTROLLERS: Tuple[str, ...] = (
    "Proposed_O1_Swerve",
    "Ablation_O1_StaticLoad",
    "Fair_Adaptive_ADMM_QP",
    "Standard_ADMM_QP",
)

CTRL_LABELS: Dict[str, str] = {
    "Proposed_O1_Swerve": "Proposed O(1) Analytic",
    "Ablation_O1_StaticLoad": "Ablation O(1) Static-Load",
    "Fair_Adaptive_ADMM_QP": "Fair Adaptive ADMM-QP",
    "Standard_ADMM_QP": "Standard ADMM-QP",
}

CTRL_COLORS: Dict[str, str] = {
    "Proposed_O1_Swerve": "#1f77b4",
    "Ablation_O1_StaticLoad": "#2ca02c",
    "Fair_Adaptive_ADMM_QP": "#f39c12",
    "Standard_ADMM_QP": "#d35400",
}


# ==============================================================================
# 2. MATHEMATICAL UTILITIES AND SCALE-INVARIANT GAIN SYNTHESIS
# ==============================================================================
def wrap_angle(a: float) -> float:
    return (a + math.pi) % (2.0 * math.pi) - math.pi


def solve_dare_exact(
    A: np.ndarray, B: np.ndarray, Q: np.ndarray, R: np.ndarray,
    tol: float = 1e-14, max_iter: int = 3000,
) -> np.ndarray:
    P = Q.copy()
    AT, BT = A.T, B.T
    for _ in range(max_iter):
        PB = P @ B
        K = np.linalg.solve(R + BT @ PB, BT @ P @ A)
        Pn = 0.5 * (Q + AT @ P @ (A - B @ K) + (Q + AT @ P @ (A - B @ K)).T)
        if np.max(np.abs(Pn - P)) < tol:
            return Pn
        P = Pn
    return P


def derive_scale_consistent_gains(spec: RobotSpec, dt: float = DT_CONTROL) -> Tuple[float, float, float, float]:
    A = np.array([
        [1.0, 0.0, dt,  0.0],
        [0.0, 1.0, 0.0, dt ],
        [0.0, 0.0, 1.0 - spec.drag_lin * dt / spec.m_nom, 0.0],
        [0.0, 0.0, 0.0, 1.0 - spec.drag_lin * dt / spec.m_nom],
    ])
    B = np.array([
        [0.5 * dt * dt / spec.m_nom, 0.0],
        [0.0, 0.5 * dt * dt / spec.m_nom],
        [dt / spec.m_nom, 0.0],
        [0.0, dt / spec.m_nom],
    ])
    m_ratio = spec.m_nom / 72.0
    Q_step = np.diag([4800.0 * m_ratio, 4800.0 * m_ratio, 280.0 * m_ratio, 280.0 * m_ratio])
    R_step = np.diag([0.002 / m_ratio, 0.002 / m_ratio])
    Qf = solve_dare_exact(A, B, Q_step, R_step)
    K = np.linalg.solve(R_step + B.T @ Qf @ B, B.T @ Qf @ A)
    kp, kv = float(K[0, 0]), float(K[0, 2])

    iz_ratio = spec.Iz_nom / 5.20
    damp_boost = max(1.0, (5.20 / spec.Iz_nom) ** 0.25)
    k_psi = 220.0 * iz_ratio
    k_r = 34.0 * iz_ratio * damp_boost
    return kp, kv, k_psi, k_r


# ==============================================================================
# 3. MULTI-CLASS SE(2) PHYSICS PLANT WITH PACEJKA TIRE MODEL & LOAD TRANSFER
# ==============================================================================
class MultiClassSwerveSE2Plant:
    def __init__(
        self, spec: RobotSpec, mass_true: float, Iz_true: float,
        motor_health: Tuple[float, ...], dt: float = DT_CONTROL, n_sub: int = N_SUBSTEPS,
    ):
        self.spec = spec
        self.m = float(mass_true)
        self.Iz = float(Iz_true)
        self.motor_health = tuple(float(h) for h in motor_health)
        self.dt = float(dt)
        self.n_sub = int(n_sub)
        self.dt_sub = self.dt / self.n_sub
        self.b_wheel = 0.015 * (spec.r_w / 0.0508)
        self.reset()

    def reset(self, x0: np.ndarray | None = None) -> None:
        self.state = np.zeros(6, dtype=float) if x0 is None else np.array(x0, dtype=float)
        self.omega_w = np.zeros(4, dtype=float)
        self.deltas = np.zeros(4, dtype=float)
        self.f_mot = np.zeros(4, dtype=float)
        self.ax_b_filt = 0.0
        self.ay_b_filt = 0.0

        px, py, psi, vx_w, vy_w, r = self.state
        c_psi, s_psi = math.cos(psi), math.sin(psi)
        vx_b = c_psi * vx_w + s_psi * vy_w
        vy_b = -s_psi * vx_w + c_psi * vy_w
        for i, (cx, cy) in enumerate(self.spec.corners):
            vbx_i = vx_b - r * cy
            vby_i = vy_b + r * cx
            self.deltas[i] = math.atan2(vby_i, vbx_i) if math.hypot(vbx_i, vby_i) > 0.05 else 0.0
            vwx_i = vbx_i * math.cos(self.deltas[i]) + vby_i * math.sin(self.deltas[i])
            self.omega_w[i] = vwx_i / self.spec.r_w

    def step(
        self, f_cmd: List[float], delta_cmd: List[float],
        mu_nom: float, mu_ice: float, ice_x_range: Tuple[float, float],
        roughness: float, ext_wrench_world: Tuple[float, float, float], z_noise: np.ndarray,
    ) -> Tuple[np.ndarray, Dict[str, Any], bool, float]:
        dts = self.dt_sub
        spec = self.spec
        alpha_acc = dts / (0.012 + dts)
        B_p, C_p, E_p = 12.0, 1.55, 0.25
        base_N = 0.25 * self.m * G
        corners = spec.corners
        any_wheel_on_ice = False
        ax_b_last = ay_b_last = slip_accum = 0.0
        Fx_ext_w, Fy_ext_w, Tz_ext = ext_wrench_world
        px, py, psi, vx_w, vy_w, r = self.state

        for _ in range(self.n_sub):
            c_psi, s_psi = math.cos(psi), math.sin(psi)
            vx_b = c_psi * vx_w + s_psi * vy_w
            vy_b = -s_psi * vx_w + c_psi * vy_w

            for i in range(4):
                h_i = self.motor_health[i]
                tau_s_i = spec.tau_steer / max(0.5, h_i)
                tau_d_i = spec.tau_drive / max(0.5, h_i)
                alpha_s = dts / (tau_s_i + dts)
                alpha_d = dts / (tau_d_i + dts)
                max_sr = spec.max_steer_rate * h_i
                err_d = wrap_angle(delta_cmd[i] - self.deltas[i])
                d_rate = max(-max_sr, min(max_sr, (alpha_s * err_d) / dts))
                self.deltas[i] = wrap_angle(self.deltas[i] + d_rate * dts)

                v_tread = self.omega_w[i] * spec.r_w
                f_stall_i = spec.f_stall * h_i
                v_free_i = spec.v_free * (0.85 + 0.15 * h_i)
                f_bemf = max(0.0, f_stall_i * (1.0 - abs(v_tread) / v_free_i))
                f_target = max(-f_bemf, min(f_bemf, float(f_cmd[i])))
                self.f_mot[i] += alpha_d * (f_target - self.f_mot[i])

            dx_n = (self.m * self.ax_b_filt * spec.h_cg) / (4.0 * spec.lx)
            dy_n = (self.m * self.ay_b_filt * spec.h_cg) / (4.0 * spec.ly)
            min_n = 0.08 * base_N
            N_loads = (
                max(min_n, base_N - dx_n - dy_n),
                max(min_n, base_N - dx_n + dy_n),
                max(min_n, base_N + dx_n - dy_n),
                max(min_n, base_N + dx_n + dy_n),
            )
            Fx_b_total = Fy_b_total = Tz_total = 0.0

            for i in range(4):
                cx, cy = corners[i]
                xw_world = px + cx * c_psi - cy * s_psi
                on_ice = ice_x_range[0] <= xw_world <= ice_x_range[1]
                if on_ice:
                    any_wheel_on_ice = True
                mu_loc = mu_ice if on_ice else mu_nom

                vbx_i = vx_b - r * cy
                vby_i = vy_b + r * cx
                cd, sd = math.cos(self.deltas[i]), math.sin(self.deltas[i])
                vwx_i = vbx_i * cd + vby_i * sd
                vwy_i = -vbx_i * sd + vby_i * cd

                v_tread = self.omega_w[i] * spec.r_w
                v_den_reg = max(math.hypot(vwx_i, vwy_i), 0.15)
                denom_k = max(abs(v_tread), v_den_reg)
                kappa = (v_tread - vwx_i) / denom_k
                lam = -vwy_i / v_den_reg
                s_comb = math.hypot(kappa, lam)
                slip_accum += s_comb

                noise_scale = roughness * 1.4 * (self.m / 72.0)
                noise_x = z_noise[2 * i] * noise_scale
                noise_y = z_noise[2 * i + 1] * noise_scale

                if s_comb > 1e-9:
                    arg = B_p * s_comb
                    mu_eff = mu_loc * math.sin(C_p * math.atan(arg - E_p * (arg - math.atan(arg))))
                    f_mag = mu_eff * N_loads[i]
                    fw_x = f_mag * (kappa / s_comb) + noise_x
                    fw_y = f_mag * (lam / s_comb) + noise_y
                else:
                    fw_x, fw_y = noise_x, noise_y

                k_tire_lin = (mu_loc * C_p * B_p * N_loads[i] * spec.r_w) / denom_k
                imp_denom = 1.0 + dts * (k_tire_lin * spec.r_w + self.b_wheel) / spec.j_w
                dw_explicit = ((self.f_mot[i] - fw_x) * spec.r_w - self.b_wheel * self.omega_w[i]) / spec.j_w
                self.omega_w[i] += (dts * dw_explicit) / imp_denom

                fb_x = fw_x * cd - fw_y * sd
                fb_y = fw_x * sd + fw_y * cd
                Fx_b_total += fb_x
                Fy_b_total += fb_y
                Tz_total += cx * fb_y - cy * fb_x

            Fx_ext_b = c_psi * Fx_ext_w + s_psi * Fy_ext_w
            Fy_ext_b = -s_psi * Fx_ext_w + c_psi * Fy_ext_w
            Fx_b_total += Fx_ext_b - spec.drag_lin * vx_b
            Fy_b_total += Fy_ext_b - spec.drag_lin * vy_b
            Tz_total += Tz_ext - spec.drag_yaw * r

            ax_b = Fx_b_total / self.m
            ay_b = Fy_b_total / self.m
            self.ax_b_filt += alpha_acc * (ax_b - self.ax_b_filt)
            self.ay_b_filt += alpha_acc * (ay_b - self.ay_b_filt)
            ax_b_last, ay_b_last = ax_b, ay_b

            ax_world = c_psi * ax_b - s_psi * ay_b
            ay_world = s_psi * ax_b + c_psi * ay_b
            alpha_yaw = Tz_total / self.Iz

            vx_w_next = vx_w + ax_world * dts
            vy_w_next = vy_w + ay_world * dts
            px += 0.5 * (vx_w + vx_w_next) * dts
            py += 0.5 * (vy_w + vy_w_next) * dts
            vx_w, vy_w = vx_w_next, vy_w_next

            r_next = r + alpha_yaw * dts
            psi = wrap_angle(psi + 0.5 * (r + r_next) * dts)
            r = r_next

        self.state = np.array([px, py, psi, vx_w, vy_w, r])
        meas = {
            "gyro_r": r + z_noise[13] * 0.006,
            "accel_body": (ax_b_last + z_noise[14] * 0.04, ay_b_last + z_noise[15] * 0.04),
            "wheel_speeds": tuple(self.omega_w[i] * spec.r_w + z_noise[16 + i] * 0.015 for i in range(4)),
            "deltas": tuple(self.deltas[i] + z_noise[20 + i] * 0.002 for i in range(4)),
        }
        return self.state.copy(), meas, any_wheel_on_ice, slip_accum / (4.0 * self.n_sub)


# ==============================================================================
# 4. SE(2) TRAJECTORY GENERATION AND COLLISION DISTURBANCE SCHEDULES
# ==============================================================================
def generate_se2_swerve_trajectory(
    T_steps: int, dt: float, v_cruise: float, amp_y: float,
    freq: float = 0.50, yaw_amp: float = 0.30,
) -> np.ndarray:
    t = np.arange(T_steps + 250) * dt
    w = 2.0 * math.pi * freq
    px = v_cruise * t
    vx = np.full_like(t, v_cruise)
    py = amp_y * np.sin(w * t)
    vy = amp_y * w * np.cos(w * t)
    psi = yaw_amp * np.sin(0.5 * w * t)
    r = yaw_amp * (0.5 * w) * np.cos(0.5 * w * t)
    return np.column_stack([px, py, psi, vx, vy, r])


def generate_collision_schedule(
    rng: np.random.Generator, spec: RobotSpec, T_steps: int, dt: float,
) -> Tuple[np.ndarray, List[int]]:
    wrenches = np.zeros((T_steps, 3), dtype=float)
    dur_steps = 10
    dur_sec = dur_steps * dt
    col_starts: List[int] = []
    col_times_sec = (float(rng.uniform(0.22, 0.30)), float(rng.uniform(1.36, 1.46)))

    for t_col in col_times_sec:
        k0 = int(round(t_col / dt))
        if k0 + dur_steps >= T_steps:
            continue
        col_starts.append(k0)
        m_opp = float(rng.uniform(0.60, 1.30)) * spec.m_nom
        v_rel_n = float(rng.uniform(0.75, 1.45))
        restitution = float(rng.uniform(0.25, 0.55))
        impact_angle = float(rng.uniform(-math.pi, math.pi))
        nx, ny = math.cos(impact_angle), math.sin(impact_angle)
        rcx = float(rng.choice([-1.0, 1.0])) * spec.lx
        rcy = float(rng.uniform(-0.85, 0.85)) * spec.ly
        r_cross_n = rcx * ny - rcy * nx
        inv_eff_mass = (1.0 / spec.m_nom) + (1.0 / m_opp) + (r_cross_n * r_cross_n) / spec.Iz_nom
        J_imp = (1.0 + restitution) * v_rel_n / inv_eff_mass
        for s in range(dur_steps):
            tau = (s + 0.5) / dur_steps
            f_mag = (0.5 * math.pi / dur_sec) * J_imp * math.sin(math.pi * tau)
            fx, fy = f_mag * nx, f_mag * ny
            tz = rcx * fy - rcy * fx
            wrenches[k0 + s, 0] += fx
            wrenches[k0 + s, 1] += fy
            wrenches[k0 + s, 2] += tz
    return wrenches, col_starts


# ==============================================================================
# 5. SOTA BASELINE SOLVER: CONDENSED ADMM-QP
# ==============================================================================
class CondensedADMMQPSolver:
    def __init__(self, spec: RobotSpec, n_iters: int = 12, rho: float = 1.5):
        self.spec = spec
        self.n_iters = n_iters
        self.rho = rho
        B = np.zeros((3, 8), dtype=float)
        for i, (cx, cy) in enumerate(spec.corners):
            B[0, 2 * i] = 1.0
            B[1, 2 * i + 1] = 1.0
            B[2, 2 * i] = -cy
            B[2, 2 * i + 1] = cx
        W_wrench = np.diag([1.0, 1.0, 1.0 / max(spec.r_sq, 1e-4)])
        self.H_inv = np.linalg.inv(B.T @ W_wrench @ B + 1e-3 * np.eye(8) + rho * np.eye(8))
        self.BTW = B.T @ W_wrench

    def solve(self, wrench_des: np.ndarray, f_caps: Tuple[float, float, float, float]) -> np.ndarray:
        rhs_base = self.BTW @ wrench_des
        z = np.zeros(8, dtype=float)
        u = np.zeros(8, dtype=float)
        for _ in range(self.n_iters):
            x = self.H_inv @ (rhs_base + self.rho * (z - u))
            v = x + u
            for i in range(4):
                vx_i, vy_i = v[2 * i], v[2 * i + 1]
                norm_i = math.hypot(vx_i, vy_i)
                cap_i = f_caps[i]
                if norm_i > cap_i and norm_i > 1e-9:
                    sc = cap_i / norm_i
                    z[2 * i], z[2 * i + 1] = vx_i * sc, vy_i * sc
                else:
                    z[2 * i], z[2 * i + 1] = vx_i, vy_i
            u += x - z
        return z


# ==============================================================================
# 6. CLOSED-LOOP EPISODE SIMULATION (FROZEN)
# ==============================================================================
def run_controller_episode(
    controller_name: str, spec: RobotSpec, traj: np.ndarray,
    wrenches: np.ndarray, noise_seq: np.ndarray,
    m_true: float, Iz_true: float, health: Tuple[float, ...],
    ice_range: Tuple[float, float], mu_ice_val: float, rough_val: float,
    col_starts: List[int], record_trace: bool = False,
) -> Tuple[Dict[str, float], Dict[str, np.ndarray] | None]:
    kp, kv, k_psi, k_r = derive_scale_consistent_gains(spec, DT_CONTROL)
    inv_4R2 = 1.0 / (4.0 * spec.r_sq)
    R_char = math.sqrt(spec.r_sq)
    beta_int = math.exp(-DT_CONTROL / 0.020)
    beta_mu_post = math.exp(-DT_CONTROL / 0.028)
    alpha_kin_filt = DT_CONTROL / (0.012 + DT_CONTROL)
    cg_aspect = spec.h_cg / max(spec.lx, 1e-3)

    is_proposed = controller_name == "Proposed_O1_Swerve"
    is_ablation = controller_name == "Ablation_O1_StaticLoad"
    is_std_admm = controller_name == "Standard_ADMM_QP"
    is_adp_admm = controller_name == "Fair_Adaptive_ADMM_QP"

    admm_solver = CondensedADMMQPSolver(spec, n_iters=12, rho=1.5) if (is_std_admm or is_adp_admm) else None

    plant = MultiClassSwerveSE2Plant(spec, m_true, Iz_true, health)
    plant.reset(traj[0])
    ox, oy, opsi, ovx, ovy, or_ = [float(v) for v in traj[0]]

    Iv_imu_x = Iv_imu_y = Iv_kin_x = Iv_kin_y = 0.0
    Iv_imu_trip0_x = Iv_imu_trip0_y = Iv_kin_trip0_x = Iv_kin_trip0_y = 0.0
    post_trip_imu_e = post_trip_kin_e = 0.0
    mu_hat = 0.84

    Iw0 = Iw1 = Iw2 = Iw3 = 0.25
    J_col_x = J_col_y = J_col_r = rho_rot_col = 0.0
    vbx_kin_prev = vby_kin_prev = ang_vel_prev = 0.0
    ax_kin_filt = ay_kin_filt = 0.0
    ax_kin_prev = ay_kin_prev = 0.0
    ds_prev = [0.0] * 4
    da_ext_prev_x = da_ext_prev_y = 0.0
    in_collision = False
    col_steps = post_col_damp_steps = 0
    in_ice_trip = False
    ice_steps_dur = 0
    sig_loss_hold = 0
    ax_b_prev = ay_b_prev = ax_b_filt = ay_b_filt = 0.0
    Fb_x_lagged = Fb_y_lagged = 0.0
    f_cmds, d_cmds = [0.0] * 4, [0.0] * 4

    T_steps = len(wrenches)
    latencies_us: List[float] = []
    pos_errs: List[float] = []
    ice_errs: List[float] = []
    col_errs: List[float] = []
    yaw_errs: List[float] = []
    slips: List[float] = []
    tp_steps = fp_steps = ice_gt_steps = non_ice_gt_steps = 0
    col_windows = [(k0, min(T_steps, k0 + 200)) for k0 in col_starts]

    if record_trace:
        tr_px = np.zeros(T_steps)
        tr_py = np.zeros(T_steps)
        tr_err_mm = np.zeros(T_steps)
        tr_yaw_deg = np.zeros(T_steps)
        tr_mu_gt = np.zeros(T_steps)
        tr_mu_hat = np.zeros(T_steps)
    else:
        tr_px = tr_py = tr_err_mm = tr_yaw_deg = tr_mu_gt = tr_mu_hat = np.empty(0)

    for k in range(T_steps):
        state_gt, meas, on_ice, step_slip = plant.step(
            f_cmds, d_cmds, 0.84, mu_ice_val, ice_range, rough_val, tuple(wrenches[k]), noise_seq[k]
        )
        px_gt, py_gt, psi_gt, _, _, _ = state_gt
        ax_b_m, ay_b_m = meas["accel_body"]
        r_gyro = meas["gyro_r"]
        ws_m = meas["wheel_speeds"]
        ds_m = meas["deltas"]
        if k == 0:
            ds_prev = list(ds_m)

        # ---- START REAL-TIME CONTROL STEP ----
        t_step_start = time.perf_counter()

        max_steer_break = 0.0
        for i in range(4):
            d_delta = abs(wrap_angle(ds_m[i] - ds_prev[i]))
            ratio_rate = (d_delta / DT_CONTROL) / max(spec.max_steer_rate, 1e-6)
            max_steer_break = max(max_steer_break, max(0.0, ratio_rate - 0.85))
        ds_prev = list(ds_m)

        vbx_list, vby_list = [0.0] * 4, [0.0] * 4
        vbx_sum = vby_sum = r_kin_num = 0.0
        for i, (cx, cy) in enumerate(spec.corners):
            cd, sd = math.cos(ds_m[i]), math.sin(ds_m[i])
            vbx_i = ws_m[i] * cd + r_gyro * cy
            vby_i = ws_m[i] * sd - r_gyro * cx
            vbx_list[i], vby_list[i] = vbx_i, vby_i
            vbx_sum += vbx_i
            vby_sum += vby_i
            r_kin_num += cx * (ws_m[i] * sd) - cy * (ws_m[i] * cd)

        r_kin = r_kin_num * inv_4R2
        curl_break = abs(r_kin - r_gyro)
        vbx_mean, vby_mean = 0.25 * vbx_sum, 0.25 * vby_sum

        vbx_k = vby_k = w_k_sum = 0.0
        for i in range(4):
            dev_i = math.hypot(vbx_list[i] - vbx_mean, vby_list[i] - vby_mean)
            wi = math.exp(-3.5 * max(0.0, dev_i - 0.14)) if (is_proposed or is_adp_admm) else 1.0
            vbx_k += wi * vbx_list[i]
            vby_k += wi * vby_list[i]
            w_k_sum += wi
        vbx_k /= max(w_k_sum, 1e-6)
        vby_k /= max(w_k_sum, 1e-6)

        ang_vel = math.atan2(vby_k, vbx_k)
        if k == 0:
            vbx_kin_prev, vby_kin_prev = vbx_k, vby_k
            ang_vel_prev = ang_vel
        d_ang_vel = abs(wrap_angle(ang_vel - ang_vel_prev))
        ang_vel_prev = ang_vel

        ax_kin_raw = (vbx_k - vbx_kin_prev) / DT_CONTROL - r_gyro * vby_k
        ay_kin_raw = (vby_k - vby_kin_prev) / DT_CONTROL + r_gyro * vbx_k
        ax_kin_filt += alpha_kin_filt * (ax_kin_raw - ax_kin_filt)
        ay_kin_filt += alpha_kin_filt * (ay_kin_raw - ay_kin_filt)
        ax_kin, ay_kin = ax_kin_filt, ay_kin_filt

        if k == 0:
            ax_kin_prev, ay_kin_prev = ax_kin, ay_kin
        d2_kin_mag = math.hypot(ax_kin - ax_kin_prev, ay_kin - ay_kin_prev)
        ax_kin_prev, ay_kin_prev = ax_kin, ay_kin
        vbx_kin_prev, vby_kin_prev = vbx_k, vby_k

        a_mag = math.hypot(ax_b_m, ay_b_m)
        da_mag = math.hypot(ax_b_m - ax_b_prev, ay_b_m - ay_b_prev)
        ax_b_prev, ay_b_prev = ax_b_m, ay_b_m

        f_cmd_mag = math.hypot(Fb_x_lagged, Fb_y_lagged)
        sc_cap = min(1.0, (0.84 * spec.m_nom * G) / max(f_cmd_mag, 1e-6))
        ax_exp = (Fb_x_lagged * sc_cap - spec.drag_lin * vbx_k) / spec.m_nom
        ay_exp = (Fb_y_lagged * sc_cap - spec.drag_lin * vby_k) / spec.m_nom
        da_ext_x = ax_b_m - ax_exp
        da_ext_y = ay_b_m - ay_exp
        da_ext_mag = math.hypot(da_ext_x, da_ext_y)

        if is_proposed or is_adp_admm:
            diff_breakpoint = (max_steer_break > 0.15) or (d_ang_vel > 0.065) or (curl_break > 0.85)
            deriv_flip = ((da_ext_x * da_ext_prev_x + da_ext_y * da_ext_prev_y) < -1.5) or (
                abs(da_ext_mag - math.hypot(da_ext_prev_x, da_ext_prev_y)) > 2.2
            )
            da_ext_prev_x, da_ext_prev_y = da_ext_x, da_ext_y
            synchronized_boundary = diff_breakpoint and deriv_flip

            mass_scale = math.sqrt(spec.m_nom / 72.0)
            accel_gap = math.hypot(ax_kin - ax_b_m, ay_kin - ay_b_m)
            centripetal_guard = max(32.0, (20.0 + 5.0 * abs(r_gyro) * math.hypot(vbx_k, vby_k)) * mass_scale)
            sign_opposition = ((ax_kin * ax_b_m + ay_kin * ay_b_m) < -3.5) and (accel_gap > 5.0)
            b_diff = (d2_kin_mag > centripetal_guard) or sign_opposition or (curl_break > 1.05)
            binary_signal_loss = b_diff and (not synchronized_boundary)

            if binary_signal_loss:
                sig_loss_hold = 6
            elif sig_loss_hold > 0:
                sig_loss_hold -= 1

            if not in_collision:
                if (a_mag > 0.96 * G or da_mag > 1.25) and da_ext_mag > 0.75 * G:
                    in_collision = True
                    col_steps = 1
                    J_col_x = da_ext_x * DT_CONTROL
                    J_col_y = da_ext_y * DT_CONTROL
                    J_col_r = r_gyro - or_
                    Iv_imu_x = Iv_imu_y = Iv_kin_x = Iv_kin_y = 0.0
                    in_ice_trip = False
                    mu_hat = 0.84
            else:
                col_steps += 1
                J_col_x += da_ext_x * DT_CONTROL
                J_col_y += da_ext_y * DT_CONTROL
                J_col_r += r_gyro - or_
                if (da_ext_mag < 0.45 * G and col_steps >= 6) or col_steps >= 35:
                    in_collision = False
                    post_col_damp_steps = min(55, max(20, col_steps * 3))
                    p_lin = spec.m_nom * R_char * math.hypot(J_col_x, J_col_y)
                    p_rot = spec.Iz_nom * abs(J_col_r)
                    rho_rot_col = p_rot / max(p_rot + p_lin, 1e-6)

            if post_col_damp_steps > 0:
                post_col_damp_steps -= 1
            else:
                rho_rot_col = 0.0

            if synchronized_boundary and not in_collision:
                Iv_imu_x = Iv_imu_y = Iv_kin_x = Iv_kin_y = 0.0
                if in_ice_trip and ice_steps_dur >= 4:
                    in_ice_trip = False
                    mu_hat = 0.84

            if not in_collision and post_col_damp_steps == 0:
                Iv_imu_x = beta_int * Iv_imu_x + ax_b_m * DT_CONTROL
                Iv_imu_y = beta_int * Iv_imu_y + ay_b_m * DT_CONTROL
                Iv_kin_x = beta_int * Iv_kin_x + ax_kin * DT_CONTROL
                Iv_kin_y = beta_int * Iv_kin_y + ay_kin * DT_CONTROL

                jx_kin, jy_kin = Iv_kin_x - Iv_imu_x, Iv_kin_y - Iv_imu_y
                kin_jump = math.hypot(jx_kin, jy_kin)
                dx_kin_d, dy_kin_d = ax_kin - ax_b_m, ay_kin - ay_b_m
                kin_rate_dot = jx_kin * dx_kin_d + jy_kin * dy_kin_d

                # Sub-linear mass-scaled threshold for low false-positive rate on light platforms
                jump_th = 0.095 * math.sqrt(mass_scale)
                gap_th = 3.0 * mass_scale
                if not in_ice_trip:
                    if (kin_jump > jump_th and kin_rate_dot > 0.022 and accel_gap > gap_th) or (
                        sig_loss_hold >= 5 and kin_jump > 0.82 * jump_th and accel_gap > 0.85 * gap_th
                    ):
                        in_ice_trip = True
                        ice_steps_dur = 1
                        Iv_imu_trip0_x, Iv_imu_trip0_y = Iv_imu_x, Iv_imu_y
                        Iv_kin_trip0_x, Iv_kin_trip0_y = Iv_kin_x, Iv_kin_y
                        post_trip_imu_e = 0.0
                        post_trip_kin_e = 0.0
                        mu_hat = 0.84
                else:
                    ice_steps_dur += 1
                    d_imu_mag = math.hypot(Iv_imu_x - Iv_imu_trip0_x, Iv_imu_y - Iv_imu_trip0_y)
                    d_kin_mag = math.hypot(Iv_kin_x - Iv_kin_trip0_x, Iv_kin_y - Iv_kin_trip0_y)
                    post_trip_imu_e += d_imu_mag * DT_CONTROL
                    post_trip_kin_e += max(d_kin_mag, d_imu_mag) * DT_CONTROL
                    if post_trip_kin_e > 1e-5:
                        ratio_raw = 0.84 * (post_trip_imu_e / post_trip_kin_e)
                        ratio_clamped = max(0.14, min(0.84, ratio_raw))
                        mu_hat = beta_mu_post * mu_hat + (1.0 - beta_mu_post) * ratio_clamped

                    if (
                        ((kin_jump < 0.60 * jump_th or kin_rate_dot <= 0.0) and sig_loss_hold == 0 and ice_steps_dur >= 4)
                        or ice_steps_dur >= 250
                    ):
                        in_ice_trip = False
                        mu_hat = 0.84
            else:
                in_ice_trip = False
                mu_hat = 0.84
        else:
            if a_mag > 0.96 * G or da_mag > 1.25:
                post_col_damp_steps = 65
            elif post_col_damp_steps > 0:
                post_col_damp_steps -= 1

        r_mid = 0.5 * (or_ + r_gyro)
        opsi_next = wrap_angle(opsi + r_mid * DT_CONTROL)
        opsi_mid = wrap_angle(opsi + 0.5 * r_mid * DT_CONTROL)
        c_mid, s_mid = math.cos(opsi_mid), math.sin(opsi_mid)
        vx_pred = ovx + (c_mid * ax_b_m - s_mid * ay_b_m) * DT_CONTROL
        vy_pred = ovy + (s_mid * ax_b_m + c_mid * ay_b_m) * DT_CONTROL

        c_n, s_n = math.cos(opsi_next), math.sin(opsi_next)
        vx_kin_w = c_n * vbx_k - s_n * vby_k
        vy_kin_w = s_n * vbx_k + c_n * vby_k
        if is_proposed or is_adp_admm:
            alpha_kin = 0.05 if in_collision else (0.10 if (in_ice_trip or sig_loss_hold > 0) else 0.36)
        else:
            alpha_kin = 0.08 if post_col_damp_steps > 0 else 0.36

        ovx_next = (1.0 - alpha_kin) * vx_pred + alpha_kin * vx_kin_w
        ovy_next = (1.0 - alpha_kin) * vy_pred + alpha_kin * vy_kin_w
        ox += 0.5 * (ovx + ovx_next) * DT_CONTROL
        oy += 0.5 * (ovy + ovy_next) * DT_CONTROL
        opsi = opsi_next
        ovx, ovy, or_ = ovx_next, ovy_next, r_gyro

        px, py, psi = ox, oy, opsi
        vx_w, vy_w, r = ovx, ovy, or_
        c_psi, s_psi = math.cos(psi), math.sin(psi)

        if is_ablation or is_std_admm:
            n0 = n1 = n2 = n3 = 0.25 * spec.m_nom * G
            w0 = w1 = w2 = w3 = 0.25
        else:
            ax_b_filt = 0.92 * ax_b_filt + 0.08 * ax_b_m
            ay_b_filt = 0.92 * ay_b_filt + 0.08 * ay_b_m
            base_N = 0.25 * spec.m_nom * G
            dx_n = (spec.m_nom * ax_b_filt * spec.h_cg) / (4.0 * spec.lx)
            dy_n = (spec.m_nom * ay_b_filt * spec.h_cg) / (4.0 * spec.ly)
            min_n = 0.08 * base_N
            n0 = max(min_n, base_N - dx_n - dy_n)
            n1 = max(min_n, base_N - dx_n + dy_n)
            n2 = max(min_n, base_N + dx_n - dy_n)
            n3 = max(min_n, base_N + dx_n + dy_n)
            inv_s = 1.0 / (n0 + n1 + n2 + n3)
            w0, w1, w2, w3 = n0 * inv_s, n1 * inv_s, n2 * inv_s, n3 * inv_s
            Iw0 = beta_int * Iw0 + (1.0 - beta_int) * w0
            Iw1 = beta_int * Iw1 + (1.0 - beta_int) * w1
            Iw2 = beta_int * Iw2 + (1.0 - beta_int) * w2
            Iw3 = beta_int * Iw3 + (1.0 - beta_int) * w3

        rx, ry, rpsi, rvx, rvy, rr = traj[k]
        nrx, nry, nrpsi, nrvx, nrvy, nrr = traj[k + 1]
        ax_ff = (nrvx - rvx) / DT_CONTROL
        ay_ff = (nrvy - rvy) / DT_CONTROL
        alpha_ff = (nrr - rr) / DT_CONTROL

        if is_proposed and not in_collision and post_col_damp_steps > 0:
            damp_lin = 1.0 + 0.35 * (1.0 - rho_rot_col)
            damp_rot = 1.0 + 0.65 * rho_rot_col
        else:
            damp_lin = damp_rot = 1.0

        Fx_w_des = spec.m_nom * ax_ff + spec.drag_lin * rvx - kp * (px - rx) - (kv * damp_lin) * (vx_w - rvx)
        Fy_w_des = spec.m_nom * ay_ff + spec.drag_lin * rvy - kp * (py - ry) - (kv * damp_lin) * (vy_w - rvy)
        Fb_x_des = c_psi * Fx_w_des + s_psi * Fy_w_des
        Fb_y_des = -s_psi * Fx_w_des + c_psi * Fy_w_des

        f_trans_mag = math.hypot(Fb_x_des, Fb_y_des)
        f_trans_cap = 0.96 * 0.84 * spec.m_nom * G
        if f_trans_mag > f_trans_cap and f_trans_mag > 1e-9:
            sc_t = f_trans_cap / f_trans_mag
            Fb_x_des *= sc_t
            Fb_y_des *= sc_t

        Tz_des = spec.Iz_nom * alpha_ff + spec.drag_yaw * rr - k_psi * wrap_angle(psi - rpsi) - (k_r * damp_rot) * (r - rr)

        ns = (n0, n1, n2, n3)
        fb_x_act = fb_y_act = 0.0

        if is_std_admm or is_adp_admm:
            mu_eff_qp = mu_hat if (is_adp_admm and in_ice_trip) else 0.84
            f_caps = tuple(mu_eff_qp * ns[i] for i in range(4))
            z_sol = admm_solver.solve(np.array([Fb_x_des, Fb_y_des, Tz_des]), f_caps)
            wheel_forces = [(z_sol[2 * i], z_sol[2 * i + 1]) for i in range(4)]
        else:
            if is_proposed and in_ice_trip:
                atten = min(0.24, 0.28 * cg_aspect) * (1.0 - mu_hat / 0.84)
                g_trans = 1.0 - atten
            else:
                g_trans = 1.0

            if is_ablation:
                Tz_net = Tz_des
                rot_weights = (0.5, 0.5, 0.5, 0.5)
            else:
                Tz_net = Tz_des
                p_diag_a = 0.5 * (Iw0 + Iw3)
                p_diag_b = 0.5 * (Iw1 + Iw2)
                p_sum = max(p_diag_a + p_diag_b, 1e-6)
                wa = 0.5 + 0.25 * (p_diag_a / p_sum - 0.5)
                wb = 1.0 - wa
                rot_weights = (wa, wb, wb, wa)

            ws = (w0, w1, w2, w3)
            wheel_forces = []
            for i in range(4):
                cx, cy = spec.corners[i]
                ft_x, ft_y = g_trans * ws[i] * Fb_x_des, g_trans * ws[i] * Fb_y_des
                rw_i = 2.0 * rot_weights[i]
                fr_x, fr_y = -cy * Tz_net * inv_4R2 * rw_i, cx * Tz_net * inv_4R2 * rw_i
                f_cap_i = 0.84 * ns[i]
                fr_norm = math.hypot(fr_x, fr_y)
                if fr_norm >= f_cap_i:
                    fxi, fyi = (fr_x * f_cap_i / fr_norm, fr_y * f_cap_i / fr_norm) if fr_norm > 1e-9 else (0.0, 0.0)
                else:
                    fxi, fyi = ft_x + fr_x, ft_y + fr_y
                    if math.hypot(fxi, fyi) > f_cap_i:
                        a_q = ft_x * ft_x + ft_y * ft_y
                        if a_q > 1e-12:
                            b_q = ft_x * fr_x + ft_y * fr_y
                            c_q = fr_norm * fr_norm - f_cap_i * f_cap_i
                            alpha_q = max(0.0, min(1.0, (-b_q + math.sqrt(max(0.0, b_q * b_q - a_q * c_q))) / a_q))
                            fxi = alpha_q * ft_x + fr_x
                            fyi = alpha_q * ft_y + fr_y
                        else:
                            fxi, fyi = fr_x, fr_y
                wheel_forces.append((fxi, fyi))

        for i in range(4):
            fxi, fyi = wheel_forces[i]
            f_bemf_i = max(20.0, spec.f_stall * (1.0 - abs(ws_m[i]) / spec.v_free))
            f_mag = math.hypot(fxi, fyi)
            if f_mag > f_bemf_i and f_mag > 1e-9:
                sc = f_bemf_i / f_mag
                fxi *= sc
                fyi *= sc
                f_mag = f_bemf_i
            if f_mag > 0.5:
                ang = math.atan2(fyi, fxi)
                err_ang = wrap_angle(ang - ds_m[i])
                if abs(err_ang) > 0.62 * math.pi:
                    ang = wrap_angle(ang + math.pi)
                    err_ang = wrap_angle(ang - ds_m[i])
                    f_mag = -f_mag
                align_gain = math.cos(err_ang) ** 2
                f_cmds[i] = f_mag * align_gain
                d_cmds[i] = ang
                fb_x_act += fxi * align_gain
                fb_y_act += fyi * align_gain
            else:
                f_cmds[i] = 0.0
                d_cmds[i] = ds_m[i]

        latencies_us.append((time.perf_counter() - t_step_start) * 1e6)
        # ---- END REAL-TIME CONTROL STEP ----

        alpha_mot = DT_CONTROL / (spec.tau_drive + DT_CONTROL)
        Fb_x_lagged += alpha_mot * (fb_x_act - Fb_x_lagged)
        Fb_y_lagged += alpha_mot * (fb_y_act - Fb_y_lagged)

        err_mm = math.hypot(px_gt - rx, py_gt - ry) * 1000.0
        yaw_deg = abs(math.degrees(wrap_angle(psi_gt - rpsi)))
        pos_errs.append(err_mm)
        yaw_errs.append(yaw_deg)
        slips.append(step_slip)

        if on_ice:
            ice_errs.append(err_mm)
            ice_gt_steps += 1
            if in_ice_trip:
                tp_steps += 1
        else:
            non_ice_gt_steps += 1
            if in_ice_trip:
                fp_steps += 1

        if any(w0_k <= k < w1_k for (w0_k, w1_k) in col_windows):
            col_errs.append(err_mm)

        if record_trace:
            tr_px[k] = px_gt
            tr_py[k] = py_gt
            tr_err_mm[k] = err_mm
            tr_yaw_deg[k] = yaw_deg
            tr_mu_gt[k] = mu_ice_val if on_ice else 0.84
            tr_mu_hat[k] = mu_hat

    # Post-collision recovery time (ms until position error settles below 150 mm or window ends)
    rec_times_ms: List[float] = []
    for k0, k1 in col_windows:
        if k1 <= k0 + 12:
            continue
        post_seg = pos_errs[k0 + 10 : k1]
        settled = next((idx for idx, val in enumerate(post_seg) if val < 150.0), len(post_seg))
        rec_times_ms.append((settled + 10) * DT_CONTROL * 1000.0)

    lat_arr = np.asarray(latencies_us[10:])  # Exclude initial warm-up samples
    metrics = {
        "mean_latency_us": float(np.mean(lat_arr)),
        "p99_latency_us": float(np.percentile(lat_arr, 99)),
        "deadline_miss_pct": float(100.0 * np.mean(lat_arr > 2000.0)),
        "peak_ice_excursion_mm": float(max(ice_errs) if ice_errs else 0.0),
        "peak_collision_deflection_mm": float(max(col_errs) if col_errs else 0.0),
        "collision_recovery_ms": float(np.mean(rec_times_ms) if rec_times_ms else 0.0),
        "rmse_mm": float(math.sqrt(np.mean(np.square(pos_errs)))),
        "peak_yaw_err_deg": float(max(yaw_errs)),
        "tire_slip_rms": float(math.sqrt(np.mean(np.square(slips)))),
        "observer_fpr": float(100.0 * fp_steps / max(non_ice_gt_steps, 1)),
        "observer_tpr": float(100.0 * tp_steps / max(ice_gt_steps, 1)),
    }

    trace_dict = None
    if record_trace:
        trace_dict = {
            "t": np.arange(T_steps) * DT_CONTROL,
            "ref_x": traj[:T_steps, 0],
            "ref_y": traj[:T_steps, 1],
            "px": tr_px,
            "py": tr_py,
            "err_mm": tr_err_mm,
            "yaw_deg": tr_yaw_deg,
            "mu_gt": tr_mu_gt,
            "mu_hat": tr_mu_hat,
        }
    return metrics, trace_dict


# ==============================================================================
# 7. ASYMPTOTIC COMPLEXITY SCALING ANALYSIS: O(1) VS. O(N^alpha)
# ==============================================================================
def run_asymptotic_scaling_benchmark(spec: RobotSpec) -> Dict[str, Any]:
    horizons = [10, 20, 40, 80, 120]
    inv_4R2 = 1.0 / (4.0 * spec.r_sq)
    corners = spec.corners
    w = (0.28, 0.22, 0.26, 0.24)
    ns = (120.0, 95.0, 110.0, 102.0)

    # Proposed O(1) analytic kernel
    t_prop: List[float] = []
    for _ in range(2000):
        t0 = time.perf_counter()
        Fb_x, Fb_y, Tz = 180.0, -90.0, 22.0
        for i in range(4):
            cx, cy = corners[i]
            ftx, fty = w[i] * Fb_x, w[i] * Fb_y
            frx, fry = -cy * Tz * inv_4R2, cx * Tz * inv_4R2
            fxi, fyi = ftx + frx, fty + fry
            cap = 0.84 * ns[i]
            nrm = math.hypot(fxi, fyi)
            if nrm > cap:
                sc = cap / nrm
                fxi *= sc
                fyi *= sc
        t_prop.append((time.perf_counter() - t0) * 1e6)
    proposed_median_us = float(np.median(t_prop[200:]))

    # Condensed MPC/ADMM-QP as a function of prediction horizon N
    offline_ms: List[float] = []
    online_us: List[float] = []
    for N in horizons:
        t0 = time.perf_counter()
        n_vars = 2 * N
        A_m = np.tril(np.ones((n_vars, n_vars))) * DT_CONTROL
        H = A_m.T @ A_m + 1.5 * np.eye(n_vars)
        H_inv = np.linalg.inv(H)
        offline_ms.append((time.perf_counter() - t0) * 1000.0)

        rhs = np.ones(n_vars) * 15.0
        t_on: List[float] = []
        for _ in range(60):
            ts = time.perf_counter()
            z = np.zeros(n_vars)
            u = np.zeros(n_vars)
            for _ in range(10):
                x = H_inv @ (rhs + 1.5 * (z - u))
                z = np.clip(x + u, -120.0, 120.0)
                u += x - z
            t_on.append((time.perf_counter() - ts) * 1e6)
        online_us.append(float(np.median(t_on[10:])))

    log_N = np.log(np.asarray(horizons, dtype=float))
    alpha_online = float(np.polyfit(log_N, np.log(np.asarray(online_us)), 1)[0])
    alpha_offline = float(np.polyfit(log_N, np.log(np.asarray(offline_ms)), 1)[0])

    return {
        "spec_name": spec.name,
        "horizons": horizons,
        "proposed_median_us": proposed_median_us,
        "offline_ms": offline_ms,
        "online_us": online_us,
        "alpha_online": alpha_online,
        "alpha_offline": alpha_offline,
    }


# ==============================================================================
# 8. PAIRED NON-PARAMETRIC STATISTICS (WILCOXON + HOLM-BONFERRONI + CLIFF'S D)
# ==============================================================================
def wilcoxon_signed_rank_p(x: np.ndarray, y: np.ndarray) -> float:
    d = np.asarray(x, dtype=float) - np.asarray(y, dtype=float)
    d = d[np.abs(d) > 1e-15]
    n = len(d)
    if n == 0:
        return 1.0
    order = np.argsort(np.abs(d))
    ranks = np.empty(n, dtype=float)
    ranks[order] = np.arange(1, n + 1, dtype=float)
    w_stat = min(float(np.sum(ranks[d > 0])), float(np.sum(ranks[d < 0])))
    mean_w = n * (n + 1) / 4.0
    std_w = math.sqrt(n * (n + 1) * (2 * n + 1) / 24.0)
    z = max(0.0, (abs(w_stat - mean_w) - 0.5) / std_w)
    return float(math.erfc(z / math.sqrt(2.0)))


def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    x_arr = np.asarray(x, dtype=float)
    y_arr = np.asarray(y, dtype=float)
    diff = x_arr[:, None] - y_arr[None, :]
    n_gt = float(np.sum(diff > 0))
    n_lt = float(np.sum(diff < 0))
    return (n_gt - n_lt) / float(len(x_arr) * len(y_arr))


def bootstrap_ci95(diff: np.ndarray, rng: np.random.Generator, n_boot: int = 4000) -> Tuple[float, float]:
    idx = rng.integers(0, len(diff), size=(n_boot, len(diff)))
    means = np.mean(diff[idx], axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def compute_holm_bonferroni_table(df: pd.DataFrame, rng: np.random.Generator) -> List[Dict[str, Any]]:
    metrics_test = [
        "mean_latency_us",
        "peak_ice_excursion_mm",
        "peak_collision_deflection_mm",
        "collision_recovery_ms",
        "rmse_mm",
        "tire_slip_rms",
    ]
    baselines = ["Ablation_O1_StaticLoad", "Fair_Adaptive_ADMM_QP", "Standard_ADMM_QP"]
    prop = df[df["controller"] == "Proposed_O1_Swerve"].sort_values("trial_global_idx").reset_index(drop=True)

    raw_rows: List[Dict[str, Any]] = []
    for base in baselines:
        b_df = df[df["controller"] == base].sort_values("trial_global_idx").reset_index(drop=True)
        for m in metrics_test:
            x = prop[m].to_numpy()
            y = b_df[m].to_numpy()
            diff = x - y
            ci_l, ci_h = bootstrap_ci95(diff, rng)
            raw_rows.append({
                "comparison": f"Proposed_vs_{base}",
                "metric": m,
                "cliffs_delta": cliffs_delta(x, y),
                "mean_diff": float(np.mean(diff)),
                "ci95_low": ci_l,
                "ci95_high": ci_h,
                "p_raw": wilcoxon_signed_rank_p(x, y),
            })

    # Holm-Bonferroni step-down adjustment
    raw_rows.sort(key=lambda r: r["p_raw"])
    m_tests = len(raw_rows)
    prev_p = 0.0
    for idx, r in enumerate(raw_rows):
        p_adj = min(1.0, max(prev_p, (m_tests - idx) * r["p_raw"]))
        r["p_holm"] = p_adj
        prev_p = p_adj
        del r["p_raw"]
    return raw_rows


# ==============================================================================
# 9. GENERATION OF THE 6 PUBLICATION FIGURES
# ==============================================================================
def generate_all_figures(
    df: pd.DataFrame, summary: Dict[str, Any],
    rep_traces: Dict[str, Dict[str, np.ndarray]], rep_ice_range: Tuple[float, float],
) -> None:
    plt.rcParams.update({
        "font.size": 9.5,
        "axes.titlesize": 10.5,
        "axes.titleweight": "bold",
        "axes.grid": True,
        "grid.alpha": 0.35,
    })

    # --- FIGURE 1: Representative Closed-Loop Trial Traces ---
    fig, axs = plt.subplots(2, 2, figsize=(13.5, 8.2), constrained_layout=True)
    fig.suptitle("Representative Closed-Loop Trial Trace (Class 3: Heavy AMR (145 kg))", fontweight="bold", fontsize=12)

    ref_tr = rep_traces["Proposed_O1_Swerve"]
    axs[0, 0].axvspan(rep_ice_range[0], rep_ice_range[1], color="#d6eaf8", alpha=0.75, label="Black-Ice Zone")
    axs[0, 0].plot(ref_tr["ref_x"], ref_tr["ref_y"], "k--", lw=1.6, label="Reference SE(2) Path")
    for c in CONTROLLERS:
        tr = rep_traces[c]
        axs[0, 0].plot(tr["px"], tr["py"], color=CTRL_COLORS[c], lw=1.5, label=CTRL_LABELS[c])
    axs[0, 0].set_title("A. Closed-Loop SE(2) Path Tracking Across Black Ice")
    axs[0, 0].set_xlabel("World X Position [m]")
    axs[0, 0].set_ylabel("World Y Position [m]")
    axs[0, 0].legend(loc="lower left", fontsize=7.5, frameon=True)

    for c in CONTROLLERS:
        tr = rep_traces[c]
        axs[0, 1].plot(tr["t"], tr["err_mm"], color=CTRL_COLORS[c], lw=1.5, label=CTRL_LABELS[c])
    axs[0, 1].set_title("B. Position Tracking Error & Post-Impact Recovery")
    axs[0, 1].set_xlabel("Time [s]")
    axs[0, 1].set_ylabel("Position Error [mm]")
    axs[0, 1].legend(loc="upper left", fontsize=7.5, frameon=True)

    axs[1, 0].plot(ref_tr["t"], ref_tr["mu_gt"], "k--", lw=1.6, label="Ground Truth Friction mu(t) [Hidden]")
    for c in ["Proposed_O1_Swerve", "Fair_Adaptive_ADMM_QP"]:
        tr = rep_traces[c]
        axs[1, 0].plot(tr["t"], tr["mu_hat"], color=CTRL_COLORS[c], lw=1.4, label=f"{CTRL_LABELS[c]} mu_hat(t)")
    axs[1, 0].set_title("C. Blind 2D Grip Observer & Inertial Shock-Gate Immunity")
    axs[1, 0].set_xlabel("Time [s]")
    axs[1, 0].set_ylabel("Friction Coefficient mu [-]")
    axs[1, 0].legend(loc="lower right", fontsize=7.5, frameon=True)

    for c in CONTROLLERS:
        tr = rep_traces[c]
        axs[1, 1].plot(tr["t"], tr["yaw_deg"], color=CTRL_COLORS[c], lw=1.5, label=CTRL_LABELS[c])
    axs[1, 1].set_title("D. Heading Error |e_psi(t)| Under Asymmetric Actuator Health")
    axs[1, 1].set_xlabel("Time [s]")
    axs[1, 1].set_ylabel("Heading Error [deg]")
    axs[1, 1].legend(loc="upper right", fontsize=7.5, frameon=True)

    fig.savefig(FIGURE_OUTPUT_DIR / "01_closed_loop_time_series_representative.png", dpi=220)
    plt.close(fig)

    # --- FIGURE 2: Global Paired Monte Carlo Distributions ---
    fig, axs = plt.subplots(2, 3, figsize=(15.0, 8.2), constrained_layout=True)
    fig.suptitle(f"Global Paired Monte Carlo Distributions Across All Robot Classes (N = {len(df)//4} Paired Trials)", fontweight="bold", fontsize=12)
    panels_f2 = [
        ("rmse_mm", "A. Trajectory Tracking RMSE [mm]", "mm"),
        ("peak_ice_excursion_mm", "B. Peak Black-Ice Excursion [mm]", "mm"),
        ("peak_collision_deflection_mm", "C. Peak Collision Deflection [mm]", "mm"),
        ("collision_recovery_ms", "D. Post-Impact Recovery Time [ms]", "ms"),
        ("tire_slip_rms", "E. Combined Tire Slip RMS [-]", "-"),
        ("observer_fpr", "F. Grip Observer False Positive Rate [%]", "%"),
    ]
    rng_jit = np.random.default_rng(42)
    for ax, (col, title, unit) in zip(axs.flat, panels_f2):
        data_boxes = [df[df["controller"] == c][col].to_numpy() for c in CONTROLLERS]
        bp = ax.boxplot(data_boxes, patch_artist=True, showfliers=False, widths=0.45)
        for idx, (patch, c) in enumerate(zip(bp["boxes"], CONTROLLERS)):
            patch.set_facecolor(CTRL_COLORS[c])
            patch.set_alpha(0.35)
            vals = data_boxes[idx]
            jit = rng_jit.uniform(-0.10, 0.10, size=len(vals))
            ax.scatter(np.full_like(vals, idx + 1) + jit, vals, color=CTRL_COLORS[c], s=14, alpha=0.65)
            ax.scatter([idx + 1], [np.mean(vals)], color="black", marker="D", s=26, zorder=5)
        ax.set_xticks(range(1, len(CONTROLLERS) + 1))
        ax.set_xticklabels([CTRL_LABELS[c].replace(" ", "\n", 1) for c in CONTROLLERS], fontsize=8)
        ax.set_title(title)
        ax.set_ylabel(unit)
    fig.savefig(FIGURE_OUTPUT_DIR / "02_global_benchmark_distributions.png", dpi=220)
    plt.close(fig)

    # --- FIGURE 3: Multi-Platform Scalability and Relative Improvement Heatmap ---
    fig, axs = plt.subplots(2, 2, figsize=(13.5, 8.4), constrained_layout=True)
    fig.suptitle("Scale-Consistent Physics Gain Synthesis Across 32 kg - 145 kg Swerve Platforms", fontweight="bold", fontsize=12)

    x_plat = np.arange(len(ROBOT_CLASSES))
    width = 0.19
    bar_specs = [
        (axs[0, 0], "rmse_mm", "A. Tracking RMSE Across Robot Masses [mm]", "upper left"),
        (axs[0, 1], "peak_collision_deflection_mm", "B. Peak Collision Deflection Across Platforms [mm]", "upper left"),
        (axs[1, 0], "collision_recovery_ms", "C. Post-Collision Settling Time [ms]", "lower left"),
    ]
    for ax, metric_name, title, leg_loc in bar_specs:
        for idx, c in enumerate(CONTROLLERS):
            means = [df[(df["platform"] == sp.name) & (df["controller"] == c)][metric_name].mean() for sp in ROBOT_CLASSES]
            sems = [df[(df["platform"] == sp.name) & (df["controller"] == c)][metric_name].std() / math.sqrt(TRIALS_PER_CLASS) for sp in ROBOT_CLASSES]
            ax.bar(x_plat + (idx - 1.5) * width, means, width, yerr=sems, capsize=3, color=CTRL_COLORS[c], label=CTRL_LABELS[c])
        ax.set_xticks(x_plat)
        ax.set_xticklabels([sp.short_label for sp in ROBOT_CLASSES])
        ax.set_title(title)
        ax.legend(loc=leg_loc, fontsize=7.5)

    # Heatmap D (Positive reduction = improvement vs. Standard ADMM-QP)
    hm_metrics = [
        ("mean_latency_us", "Mean Latency"),
        ("rmse_mm", "Tracking RMSE"),
        ("peak_ice_excursion_mm", "Black-Ice Excursion"),
        ("peak_collision_deflection_mm", "Collision Deflection"),
        ("collision_recovery_ms", "Recovery Time"),
        ("tire_slip_rms", "Tire Slip RMS"),
    ]
    hm_mat = np.zeros((len(hm_metrics), len(ROBOT_CLASSES)))
    for r_i, (m_key, _) in enumerate(hm_metrics):
        for c_i, sp in enumerate(ROBOT_CLASSES):
            v_prop = df[(df["platform"] == sp.name) & (df["controller"] == "Proposed_O1_Swerve")][m_key].mean()
            v_sota = df[(df["platform"] == sp.name) & (df["controller"] == "Standard_ADMM_QP")][m_key].mean()
            hm_mat[r_i, c_i] = 100.0 * (v_sota - v_prop) / max(v_sota, 1e-9)

    im = axs[1, 1].imshow(hm_mat, cmap="YlGnBu", vmin=-15.0, vmax=75.0, aspect="auto")
    axs[1, 1].set_xticks(range(len(ROBOT_CLASSES)))
    axs[1, 1].set_xticklabels([sp.short_label for sp in ROBOT_CLASSES])
    axs[1, 1].set_yticks(range(len(hm_metrics)))
    axs[1, 1].set_yticklabels([lbl for _, lbl in hm_metrics])
    axs[1, 1].set_title("D. Zero-Retuning Relative Improvement Heatmap [%]")
    for r_i in range(len(hm_metrics)):
        for c_i in range(len(ROBOT_CLASSES)):
            val = hm_mat[r_i, c_i]
            txt_color = "white" if val > 40.0 else "black"
            axs[1, 1].text(c_i, r_i, f"{val:+.1f}%", ha="center", va="center", color=txt_color, fontweight="bold", fontsize=8.5)
    cbar = fig.colorbar(im, ax=axs[1, 1])
    cbar.set_label("Reduction vs. Standard ADMM-QP [%]")

    fig.savefig(FIGURE_OUTPUT_DIR / "03_multi_platform_scalability_heatmap.png", dpi=220)
    plt.close(fig)

    # --- FIGURE 4: Real-Time Latency and Asymptotic Complexity ---
    fig, axs = plt.subplots(1, 2, figsize=(13.5, 5.2), constrained_layout=True)
    fig.suptitle("Computational Complexity & Hard Real-Time Determinism Analysis", fontweight="bold", fontsize=12)

    x_c = np.arange(len(CONTROLLERS))
    m_lats = [df[df["controller"] == c]["mean_latency_us"].mean() for c in CONTROLLERS]
    p99_lats = [df[df["controller"] == c]["p99_latency_us"].mean() for c in CONTROLLERS]
    axs[0].bar(x_c - 0.18, m_lats, 0.35, color="#1f618d", label="Mean Step Latency [us]")
    axs[0].bar(x_c + 0.18, p99_lats, 0.35, color="#f39c12", label="p99 Tail Latency [us]")
    axs[0].axhline(2000.0, color="red", ls="--", lw=1.6, label="500 Hz Hard Deadline (2000 us)")
    axs[0].axhline(1000.0, color="purple", ls=":", lw=1.5, label="1 kHz Control Budget (1000 us)")
    axs[0].set_yscale("log")
    axs[0].set_xticks(x_c)
    axs[0].set_xticklabels([CTRL_LABELS[c].replace(" ", "\n", 1) for c in CONTROLLERS])
    axs[0].set_ylabel("Execution Time per Control Step [us] (log scale)")
    axs[0].set_title("A. Real-Time Loop Latency & 500 Hz Deadline Compliance")
    axs[0].legend(loc="center left", fontsize=8)

    asymp = summary["asymptotic_scaling"]
    hs = asymp["horizons"]
    axs[1].loglog(hs, [asymp["proposed_median_us"]] * len(hs), "o-", lw=2.0, label="Proposed O(1) Online Step (alpha = 0.00)")
    axs[1].loglog(hs, asymp["online_us"], "s-", lw=2.0, label=f"Condensed ADMM-QP Online Solve (alpha = {asymp['alpha_online']:.2f})")
    axs[1].loglog(hs, [v * 1000.0 for v in asymp["offline_ms"]], "^--", lw=1.8, label=f"Condensed ADMM-QP Offline Setup (alpha = {asymp['alpha_offline']:.2f})")
    axs[1].axhline(2000.0, color="red", ls=":", lw=1.6, label="500 Hz Control Period (2000 us)")
    axs[1].set_xlabel("Prediction Horizon Length N [steps]")
    axs[1].set_ylabel("Execution Time [us] (log-log scale)")
    axs[1].set_title(f"B. Asymptotic Complexity Scaling ({asymp['spec_name']})")
    axs[1].legend(loc="upper left", fontsize=8)

    fig.savefig(FIGURE_OUTPUT_DIR / "04_realtime_latency_and_asymptotic_scaling.png", dpi=220)
    plt.close(fig)

    # --- FIGURE 5: Statistical Forest Plot (Holm-Bonferroni + Cliff's Delta) ---
    fig, axs = plt.subplots(1, 3, figsize=(15.0, 5.2), constrained_layout=True)
    fig.suptitle("Holm-Bonferroni Adjusted Paired Differences, 95% Bootstrap CIs & Cliff's Delta Effect Sizes", fontweight="bold", fontsize=12)

    hb_df = pd.DataFrame(summary["holm_bonferroni_pairwise"])
    comps = [
        ("Proposed_vs_Ablation_O1_StaticLoad", "A. Proposed O(1) vs. Ablation (Static Load)"),
        ("Proposed_vs_Fair_Adaptive_ADMM_QP", "B. Proposed O(1) vs. Fair Adaptive ADMM-QP"),
        ("Proposed_vs_Standard_ADMM_QP", "C. Proposed O(1) vs. Standard ADMM-QP"),
    ]
    metric_order = [
        "mean_latency_us", "peak_ice_excursion_mm", "peak_collision_deflection_mm",
        "collision_recovery_ms", "rmse_mm", "tire_slip_rms",
    ]
    for ax, (comp_key, title) in zip(axs, comps):
        sub = hb_df[hb_df["comparison"] == comp_key].set_index("metric").loc[metric_order]
        y_pos = np.arange(len(metric_order))
        ax.axvline(0.0, color="black", ls="--", lw=1.1)
        for idx, (m_name, row) in enumerate(sub.iterrows()):
            color_pt = "#117a65" if row["mean_diff"] < 0 else "#34495e"
            ax.errorbar(
                row["mean_diff"], idx,
                xerr=[[row["mean_diff"] - row["ci95_low"]], [row["ci95_high"] - row["mean_diff"]]],
                fmt="o", color=color_pt, ecolor=color_pt, elinewidth=1.8, capsize=3.5,
            )
            ax.text(
                row["mean_diff"], idx + 0.22,
                f"{row['mean_diff']:+.1f} | d={row['cliffs_delta']:+.2f}",
                ha="center", va="bottom", fontsize=7.5, color=color_pt, fontweight="bold",
            )
        ax.set_yticks(y_pos)
        ax.set_yticklabels(metric_order)
        ax.set_xlabel("Paired Difference (Proposed - Baseline) [95% Bootstrap CI]")
        ax.set_title(title)

    fig.savefig(FIGURE_OUTPUT_DIR / "05_statistical_forest_plot_effect_sizes.png", dpi=220)
    plt.close(fig)

    # --- FIGURE 6: Executive Benchmark Dashboard ---
    fig, axs = plt.subplots(2, 3, figsize=(14.5, 8.0), constrained_layout=True)
    fig.suptitle(f"O(1) Dynamic-Load & Grip-Aware Swerve Control — Executive Benchmark Dashboard (N = {len(df)//4} Trials)", fontweight="bold", fontsize=12)

    g_sum = summary["global_summary"]
    prop_s = g_sum["Proposed_O1_Swerve"]
    sota_s = g_sum["Standard_ADMM_QP"]
    speedup = sota_s["mean_latency_us"]["mean"] / prop_s["mean_latency_us"]["mean"]
    rmse_red = 100.0 * (sota_s["rmse_mm"]["mean"] - prop_s["rmse_mm"]["mean"]) / sota_s["rmse_mm"]["mean"]
    col_red = 100.0 * (sota_s["peak_collision_deflection_mm"]["mean"] - prop_s["peak_collision_deflection_mm"]["mean"]) / sota_s["peak_collision_deflection_mm"]["mean"]

    kpi_cards = [
        (axs[0, 0], "COMPUTE SPEEDUP", f"{speedup:.1f}x", "#0077c2",
         f"{prop_s['mean_latency_us']['mean']:.2f} us vs {sota_s['mean_latency_us']['mean']:.1f} us\n{prop_s['deadline_miss_pct']['mean']:.2f}% deadline misses @ 500 Hz"),
        (axs[0, 1], "TRACKING RMSE REDUCTION", f"{rmse_red:+.1f}%", "#009e60",
         f"{prop_s['rmse_mm']['mean']:.1f} mm vs {sota_s['rmse_mm']['mean']:.1f} mm\nBlack-Ice Max: {prop_s['peak_ice_excursion_mm']['mean']:.1f} mm"),
        (axs[0, 2], "COLLISION DEFLECTION REDUCTION", f"{col_red:+.1f}%", "#7d3c98",
         f"{prop_s['peak_collision_deflection_mm']['mean']:.1f} mm vs {sota_s['peak_collision_deflection_mm']['mean']:.1f} mm\nSettling: {prop_s['collision_recovery_ms']['mean']:.1f} ms"),
    ]
    for ax, title, big_num, col_hex, sub_txt in kpi_cards:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.text(0.5, 0.80, title, ha="center", va="center", fontweight="bold", fontsize=9.5, color="#2c3e50")
        ax.text(0.5, 0.48, big_num, ha="center", va="center", fontweight="bold", fontsize=24, color=col_hex)
        ax.text(0.5, 0.18, sub_txt, ha="center", va="center", fontsize=8.5, color="#566573")

    # Bottom-Left Panel: Global Comparison Bars
    g_cols = ["rmse_mm", "peak_ice_excursion_mm", "peak_collision_deflection_mm", "collision_recovery_ms"]
    g_lbls = ["RMSE\n[mm]", "Ice Max\n[mm]", "Col Defl\n[mm]", "Rec Time\n[ms]"]
    xg = np.arange(len(g_cols))
    for idx, c in enumerate(CONTROLLERS):
        vals = [g_sum[c][k_m]["mean"] for k_m in g_cols]
        axs[1, 0].bar(xg + (idx - 1.5) * 0.19, vals, 0.19, color=CTRL_COLORS[c], label=CTRL_LABELS[c])
    axs[1, 0].set_xticks(xg)
    axs[1, 0].set_xticklabels(g_lbls)
    axs[1, 0].set_title("Global Error & Recovery Comparison")
    axs[1, 0].legend(loc="upper left", fontsize=7.2)

    # Bottom-Center Panel: Compute vs. Accuracy Pareto Frontier
    for c in CONTROLLERS:
        lx_v = g_sum[c]["mean_latency_us"]["mean"]
        ry_v = g_sum[c]["rmse_mm"]["mean"]
        axs[1, 1].scatter([lx_v], [ry_v], s=110, color=CTRL_COLORS[c], edgecolors="black", zorder=5)
        axs[1, 1].annotate(
            f"{CTRL_LABELS[c]}\n({lx_v:.1f} us, {ry_v:.1f} mm)",
            (lx_v, ry_v), textcoords="offset points", xytext=(6, 4), fontsize=7.5, fontweight="bold",
        )
    axs[1, 1].set_xscale("log")
    axs[1, 1].set_xlabel("Mean Compute Latency [us] (log scale)")
    axs[1, 1].set_ylabel("Mean Tracking RMSE [mm]")
    axs[1, 1].set_title("Compute-Accuracy Pareto Frontier (Bottom-Left = Optimal)")

    # Bottom-Right Panel: Scale Invariance Across Mass Classes
    for c in CONTROLLERS:
        y_sc = [df[(df["platform"] == sp.name) & (df["controller"] == c)]["rmse_mm"].mean() for sp in ROBOT_CLASSES]
        axs[1, 2].plot(["32 kg Mini", "72 kg Std", "145 kg AMR"], y_sc, "o-", lw=2.0, color=CTRL_COLORS[c], label=CTRL_LABELS[c])
    axs[1, 2].set_ylabel("Tracking RMSE [mm]")
    axs[1, 2].set_title("Zero-Retuning Scale Invariance Across Mass Classes")
    axs[1, 2].legend(loc="upper left", fontsize=7.5)

    fig.savefig(FIGURE_OUTPUT_DIR / "06_executive_summary_dashboard.png", dpi=220)
    plt.close(fig)


# ==============================================================================
# 10. MAIN BENCHMARK ORCHESTRATION
# ==============================================================================
def main() -> None:
    t_start_all = time.perf_counter()
    rng = np.random.default_rng(SEED_MASTER)
    rows: List[Dict[str, Any]] = []
    rep_traces: Dict[str, Dict[str, np.ndarray]] = {}
    rep_ice_range: Tuple[float, float] = (1.2, 2.0)

    trial_global_idx = 0
    for spec in ROBOT_CLASSES:
        for ep in range(TRIALS_PER_CLASS):
            v_c = float(rng.uniform(*spec.v_cruise_range))
            amp_y = float(rng.uniform(*spec.amp_y_range))
            traj = generate_se2_swerve_trajectory(T_STEPS_EPISODE, DT_CONTROL, v_c, amp_y)
            wrenches, col_starts = generate_collision_schedule(rng, spec, T_STEPS_EPISODE, DT_CONTROL)
            noise_seq = rng.standard_normal((T_STEPS_EPISODE, 24))

            m_true = spec.m_nom * float(rng.uniform(0.92, 1.12))
            Iz_true = spec.Iz_nom * float(rng.uniform(0.92, 1.12))
            health = tuple(float(rng.uniform(0.82, 1.0)) for _ in range(4))

            ice_x0 = float(rng.uniform(1.10, 1.35))
            ice_range = (ice_x0, ice_x0 + float(rng.uniform(0.75, 0.95)))
            mu_ice_val = float(rng.choice([0.18, 0.18, 0.10]))
            rough_val = 0.70 if mu_ice_val < 0.15 else 0.35

            is_rep = (spec.name == "Class3_HeavyAMR_145kg") and (ep == 0)
            if is_rep:
                rep_ice_range = ice_range

            for ctrl in CONTROLLERS:
                m_dict, tr = run_controller_episode(
                    ctrl, spec, traj, wrenches, noise_seq,
                    m_true, Iz_true, health, ice_range, mu_ice_val, rough_val,
                    col_starts, record_trace=is_rep,
                )
                if is_rep and tr is not None:
                    rep_traces[ctrl] = tr

                row = {
                    "trial_global_idx": trial_global_idx,
                    "platform": spec.name,
                    "episode_idx": ep,
                    "controller": ctrl,
                    "mu_ice_true": mu_ice_val,
                    **m_dict,
                }
                rows.append(row)

            trial_global_idx += 1

    df = pd.DataFrame(rows)
    df.to_csv(BENCHMARK_OUTPUT_DIR / "metrics_per_trial.csv", index=False)

    metric_cols = [
        "mean_latency_us", "p99_latency_us", "deadline_miss_pct",
        "peak_ice_excursion_mm", "peak_collision_deflection_mm",
        "collision_recovery_ms", "rmse_mm", "peak_yaw_err_deg",
        "tire_slip_rms", "observer_fpr", "observer_tpr",
    ]

    global_summary: Dict[str, Any] = {}
    for c in CONTROLLERS:
        sub = df[df["controller"] == c]
        global_summary[c] = {
            m: {
                "mean": float(sub[m].mean()),
                "std": float(sub[m].std()),
                "median": float(sub[m].median()),
            }
            for m in metric_cols
        }

    per_platform_summary: Dict[str, Any] = {}
    for sp in ROBOT_CLASSES:
        per_platform_summary[sp.name] = {}
        for c in CONTROLLERS:
            sub = df[(df["platform"] == sp.name) & (df["controller"] == c)]
            per_platform_summary[sp.name][c] = {
                m: {
                    "mean": float(sub[m].mean()),
                    "std": float(sub[m].std()),
                }
                for m in metric_cols
            }

    asymp_summary = run_asymptotic_scaling_benchmark(ROBOT_CLASSES[2])
    hb_summary = compute_holm_bonferroni_table(df, rng)

    summary = {
        "meta": {
            "dt_control_s": DT_CONTROL,
            "control_freq_hz": CONTROL_FREQ_HZ,
            "physics_substeps": N_SUBSTEPS,
            "trials_per_class": TRIALS_PER_CLASS,
            "total_paired_trials": TRIALS_PER_CLASS * len(ROBOT_CLASSES),
        },
        "global_summary": global_summary,
        "per_platform_summary": per_platform_summary,
        "holm_bonferroni_pairwise": hb_summary,
        "asymptotic_scaling": asymp_summary,
    }

    with (BENCHMARK_OUTPUT_DIR / "metrics_summary.json").open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    np.savez_compressed(
        BENCHMARK_OUTPUT_DIR / "benchmark_cache.npz",
        rmse_mm=df["rmse_mm"].to_numpy(),
        latency_us=df["mean_latency_us"].to_numpy(),
        ice_mm=df["peak_ice_excursion_mm"].to_numpy(),
        col_mm=df["peak_collision_deflection_mm"].to_numpy(),
    )

    generate_all_figures(df, summary, rep_traces, rep_ice_range)
    print(f"Frozen benchmark completed in {time.perf_counter() - t_start_all:.2f} s. Metrics saved to benchmark_data/ and figures saved to docs/images/benchmark_suite/.")


if __name__ == "__main__":
    main()

Benchmark congelado completado en 199.91 s. Archivos guardados en el directorio actual.


In [3]:
import os, glob; print("Carpeta actual:", os.getcwd(), "\nArchivos:", [os.path.abspath(f) for f in sorted(glob.glob("*"))])

Carpeta actual: c:\Users\HP ENVY\Downloads\kinodynamic-swerve-simulation-control-main 
Archivos: ['c:\\Users\\HP ENVY\\Downloads\\kinodynamic-swerve-simulation-control-main\\01_closed_loop_time_series_representative.png', 'c:\\Users\\HP ENVY\\Downloads\\kinodynamic-swerve-simulation-control-main\\02_global_benchmark_distributions.png', 'c:\\Users\\HP ENVY\\Downloads\\kinodynamic-swerve-simulation-control-main\\03_multi_platform_scalability_heatmap.png', 'c:\\Users\\HP ENVY\\Downloads\\kinodynamic-swerve-simulation-control-main\\04_realtime_latency_and_asymptotic_scaling.png', 'c:\\Users\\HP ENVY\\Downloads\\kinodynamic-swerve-simulation-control-main\\05_statistical_forest_plot_effect_sizes.png', 'c:\\Users\\HP ENVY\\Downloads\\kinodynamic-swerve-simulation-control-main\\06_executive_summary_dashboard.png', 'c:\\Users\\HP ENVY\\Downloads\\kinodynamic-swerve-simulation-control-main\\Final obstacle calibration tweaks.png', 'c:\\Users\\HP ENVY\\Downloads\\kinodynamic-swerve-simulation-cont